# ConvLSTM SIPETA v8: rumus LSTM ditulis sendiri + jumlah layer ikut grid search

**Apa yang sama dengan v7.** Data dan labelnya tidak berubah: peta tahunan dari **komposit mayoritas** semua tanggal, lalu **aturan konsistensi waktu** (piksel yang terbangun dua tahun berturut-turut dianggap tetap terbangun; aturan ini hanya melihat tahun sebelumnya). Cache data v7 dipakai ulang, jadi tahap data tidak perlu dihitung lagi. Pembagian tahun, early stopping, metrik evaluasi, dan prediksi 2026–2027 juga sama.

**Apa yang baru di v8:**

1. **Layer ConvLSTM ditulis sendiri** (`ConvLSTMManual`, bagian 3). Rumus gate, cell state, hidden state (Persamaan 2.39–2.44), perulangan tahun, dan state awal nol ditulis di kode. Yang tetap dari TensorFlow/Keras: operasi konvolusi (`tf.nn.conv2d`), penghitungan gradien otomatis, optimizer Adam, BatchNormalization, Conv2D output, dan proses `fit`.
2. **Sel pembuktian** (bagian 3b): bobot dari `ConvLSTM2D` Keras disalin ke layer manual, lalu keluarannya dibandingkan. Hasilnya harus sama; kalau tidak, sel berhenti dengan pesan error.
3. **Jumlah layer ikut grid search** (`n_layers`: 1, 2, 3). Di v7 nilainya ditetapkan 2 tanpa diuji. Alasannya ada di tabel bagian 0. Bagian 6 menampilkan kombinasi terbaik untuk tiap jumlah layer.
4. **Perbandingan v7 vs v8** di akhir. Labelnya sama persis, jadi angkanya bisa dibandingkan langsung.

**Isi notebook ini.** Input: peta biner hasil Random Forest per kecamatan (0 = nonterbangun, 1 = terbangun, 255 = nodata). Output: kombinasi hyperparameter ConvLSTM terbaik, model terlatih, evaluasinya, dan peta serta luas prediksi 2026 dan 2027.

**Alurnya ada 6 langkah:**

| No | Langkah | Hasil |
|---|---|---|
| 1 | Siapkan data: 1 peta per tahun per kecamatan (komposit mayoritas + konsistensi waktu), di-resize 64×64, disimpan ke cache | `ANNUAL`, tabel stabilitas label |
| 2 | Bagi tahun jadi train / validasi / test (urut waktu) | train 2020–2022, val 2023, test 2024–2025 |
| 3 | Grid search: tiap kombinasi dilatih di train, dinilai di validasi (bisa dilanjut kalau Colab putus) | `gridsearch_log.csv` |
| 4 | Ambil kombinasi terbaik, uji di data test 2024 dan 2025 | metrik + confusion matrix |
| 5 | Latih model final dengan data sampai 2025, prediksi 2026 dan 2027 | model `.keras`, CSV luas, peta `.npz` |
| 6 | Plot peta per kecamatan, lalu bandingkan dengan v7 | PNG, tabel v7 vs v8 |

**Metrik yang dihitung di setiap evaluasi** (grid search, test):
- *Peta (piksel)*: confusion matrix (TP, FP, FN, TN), akurasi, presisi, recall, F1, IoU, FoM.
- *Luas per kecamatan (km²)*: MAE, RMSE, MAPE, R², dan bias total.
- Semua angka di atas juga dihitung untuk **baseline persistensi** (peta tahun lalu disalin apa adanya).

**Jalankan dengan GPU** (Runtime → Change runtime type → T4 GPU). Jalankan semua sel berurutan dari atas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Konfigurasi

Semua pengaturan ada di satu sel ini, jadi tidak perlu mengubah sel lain. Pengaturan ini sejajar dengan form *Latih Model* di aplikasi:

| Form aplikasi | Variabel | Masuk grid search? |
|---|---|---|
| Nama model | `MODEL_NAME` | tidak (nama file output) |
| Wilayah / Kecamatan | `SELECTED_KOTA` / `SELECTED_KECAMATAN` | tidak |
| Data latih | `RF_OUTPUT_ROOT` | tidak |
| Train / Val / Test ratio | `TRAIN_RATIO`, `VAL_RATIO`, `TEST_RATIO` | tidak (dipotong urut tahun) |
| Epoch, Batch size | `MAX_EPOCHS`, `BATCH_SIZE` | tidak (epoch dibatasi early stopping) |
| Sequence length, Jumlah layers, Kernel size, Dropout, Learning rate, Optimizer | `PARAM_GRID` | ya |

`filters` (jumlah filter per layer) juga masuk grid. Parameter ini tidak ada di form, padahal cukup berpengaruh, jadi sebaiknya ditambahkan ke form aplikasi.

**Alasan nilai di `PARAM_GRID`:**

| Parameter | Nilai | Alasan |
|---|---|---|
| `filters` | 16, 40, 64 | 40 dipakai Boulila dkk. (2021) untuk prediksi ekspansi kota dengan ConvLSTM (4 layer ConvLSTM, masing-masing 40 filter, kernel 3×3). 16 dan 64 untuk melihat apakah model lebih kecil atau lebih besar lebih cocok dengan data sesedikit ini. |
| `kernel_size` | 3×3, 5×5 | 3×3 dipakai Boulila dkk. (2021). Shi dkk. (2015) menjelaskan kernel lebih besar menangkap perubahan yang menjalar lebih jauh. Ukuran 1 sel 64×64 berbeda tiap kecamatan (lebar raster asli ÷ 64); untuk kecamatan sedang sekitar 100 m. |
| `learning_rate` | 0,001; 0,0005 | 0,001 adalah default Adam dan nilai Boulila dkk.; 0,0005 untuk jaga-jaga kalau training tidak stabil. |
| `sequence_length` | 2, 3, 4 | Boulila dkk. memakai 3 citra. Maksimum 4 supaya tiap kecamatan masih punya cukup jendela latih dari 10 tahun data. |
| `n_layers` | 1, 2, 3 | Tidak ada angka baku di literatur. Shi dkk. (2015, bag. 4.1) menguji 1, 2, dan 3 layer: lebih dalam sedikit lebih baik, dengan selisih 2→3 layer kecil, padahal data latih mereka 10.000 urutan. Boulila dkk. (2021) memakai 4 layer tanpa menguji jumlah lain, O'Neill dkk. (2025) memakai 1 layer, dan Durrani dkk. (2023) menjadikan jumlah layer hyperparameter (terbaik 2 layer untuk kasus mereka). Data kita hanya 147 urutan latih, jadi jumlah layer diuji, bukan ditetapkan. |
| `dropout` | 0,0 | Default model lama. Layer manual mendukung dropout input seperti `ConvLSTM2D`. Coba `[0.0, 0.2]` kalau loss train turun tapi loss validasi naik (overfit). |
| `optimizer` | adam | Sama dengan model lama dan Boulila dkk. |

Total 3 × 2 × 2 × 3 × 3 = 108 kombinasi. Di v7 (2 layer, batas 100 epoch) satu kombinasi butuh sekitar 1,2–6,9 menit di GPU. Kombinasi 1 layer lebih ringan dan 3 layer kira-kira 1,7 kali lebih berat, jadi perkiraan kasar seluruh grid sekitar 4–6 jam. Kecepatan layer manual di GPU belum diukur; ukur dulu dengan 2 kombinasi (lihat bagian 5). Grid bisa dilanjut lintas sesi Colab.

**Kenapa `MAX_EPOCHS = 100` dan `PATIENCE = 15`:** di run sebelumnya (batas 30 epoch, patience 6), sebagian besar kombinasi berhenti di epoch 1–2 dengan `val_loss` ≈ 0,69 (= menebak 0,5 di semua piksel), sedangkan kombinasi yang berhasil justru masih membaik saat menyentuh epoch 30. Early stopping tetap menghentikan training kalau `val_loss` tidak membaik 15 epoch berturut-turut.

**Semua hasil disimpan di subfolder baru** `convlstm_output/v8_mayoritas_konsisten_semua_p64/` (log grid, model, CSV, gambar). Nama subfolder ikut berubah kalau `COMPOSITE_MODE`, `TEMPORAL_CONSISTENCY`, `COMPOSITE_MONTHS`, atau `PATCH_SIZE` diubah, jadi hasil dari pengaturan berbeda tidak tercampur. Hasil v7 tidak disentuh; v8 hanya membacanya untuk perbandingan. Cache data (`convlstm_annual_cache_...pkl`) sama dengan v7, jadi dipakai ulang.

**`START_FROM_EPOCH = 20`:** di v6, 8 dari 21 kombinasi berhenti di epoch 1–2 dengan IoU 0 (menebak nonterbangun di semua piksel), padahal kombinasi yang berhasil mencapai `val_loss` terbaiknya di epoch 33–42. Dengan pengaturan ini, early stopping baru menilai setelah epoch 20, jadi model punya waktu keluar dari fase awal tersebut. Kombinasi yang berhasil di v6 tidak terpengaruh karena epoch terbaiknya sudah di atas 20.

In [ ]:
import os, re, time, glob, json, pickle, itertools
import numpy as np
import pandas as pd
import rasterio

# ---------------- lokasi data ----------------
SKRIPSI_DIR = "/content/drive/MyDrive/Colab Notebooks/skripsi"
RF_OUTPUT_ROOT = os.path.join(SKRIPSI_DIR, "kecamatan_masked_rf_classified")   # Data latih

# ---------------- wilayah ----------------
MODEL_NAME = "convlstm_sipeta"
SELECTED_KOTA = ["kota_tangerang", "kota_tangerang_selatan", "kabupaten_tangerang"]   # Wilayah
SELECTED_KECAMATAN = None          # None = semua; atau mis. ["serpong", "pinang"]

# ---------------- data ----------------
LABEL_NODATA = 255
PIXEL_AREA_KM2 = (10 * 10) / 1_000_000      # 1 piksel Sentinel-2 10 m = 0,0001 km2
PATCH_SIZE = 64                             # ukuran seragam tiap kecamatan setelah resize
YEARS_ALL = list(range(2016, 2026))         # data aktual 2016-2025
LAST_OBSERVED_YEAR = YEARS_ALL[-1]
N_FORECAST = 2                              # 2026 dan 2027
COMPOSITE_MONTHS = None                     # None = semua bulan; mis. (5, 10) = hanya Mei-Oktober

# ---------------- cara membuat peta tahunan (baru di v7) ----------------
COMPOSITE_MODE = "mayoritas"      # "mayoritas" (v7) atau "tanggal_terbaik" (cara v6: 1 tanggal per tahun)
MAJORITY_MIN_FRAC = 0.5           # terbangun kalau LEBIH DARI 50% tanggal valid tahun itu terbangun
TEMPORAL_CONSISTENCY = True       # terbangun 2 tahun berturut-turut -> tetap terbangun (sama dengan v7)

# Semua hasil (log grid, model, CSV, gambar) masuk ke subfolder sesuai pengaturan data,
# supaya hasil dari pengaturan yang berbeda tidak tercampur.
DATA_TAG = (f"{COMPOSITE_MODE}_{'konsisten' if TEMPORAL_CONSISTENCY else 'mentah'}_"
            + ("semua" if COMPOSITE_MONTHS is None else f"bulan{COMPOSITE_MONTHS[0]}-{COMPOSITE_MONTHS[1]}")
            + f"_p{PATCH_SIZE}")
OUTPUT_DIR = os.path.join(SKRIPSI_DIR, "convlstm_output", f"v8_{DATA_TAG}")             # folder baru, v7 tidak tertimpa
V7_OUTPUT_DIR = os.path.join(SKRIPSI_DIR, "convlstm_output", f"v7_{DATA_TAG}")          # untuk perbandingan
GRID_LOG_PATH = os.path.join(OUTPUT_DIR, "gridsearch_log.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- pembagian data (urut tahun) ----------------
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.5, 0.17, 0.33

# ---------------- training ----------------
MAX_EPOCHS = 100
PATIENCE = 15
START_FROM_EPOCH = 20              # early stopping baru mulai memantau setelah epoch ini
BATCH_SIZE = 8
THRESHOLD = 0.5
RANDOM_STATE = 42
MAX_MINUTES_PER_SESSION = None     # mis. 150: grid berhenti rapi sebelum sesi Colab habis

# ---------------- grid search ----------------
PARAM_GRID = {
    "filters": [16, 40, 64],
    "kernel_size": [3, 5],
    "learning_rate": [0.001, 0.0005],
    "sequence_length": [2, 3, 4],
    "n_layers": [1, 2, 3],          # jumlah layer ConvLSTM (v7: hanya 2)
    "dropout": [0.0],
    "optimizer": ["adam"],
}

# Isi kalau mau melewati grid dan langsung melatih SATU konfigurasi, mis.:
# MANUAL_CONFIG = dict(filters=16, kernel_size=5, learning_rate=0.001, sequence_length=4,
#                      n_layers=2, dropout=0.0, optimizer="adam", epochs=30)
MANUAL_CONFIG = None

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
!pip install scikit-image --quiet
from skimage.transform import resize as sk_resize
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", [g.name for g in gpus] if gpus else "TIDAK ADA (Runtime > Change runtime type > T4 GPU)")

## 1. Siapkan data: 1 peta per tahun per kecamatan

ConvLSTM butuh 1 peta per langkah waktu, sedangkan tiap tahun punya beberapa tanggal. Ada dua cara membuat peta tahunan:

- **`"tanggal_terbaik"` (cara v6):** tahun diwakili satu tanggal dengan piksel valid terbanyak. Hasilnya bergantung pada kondisi permukaan di hari itu (sawah tergenang atau bera, tanah basah atau kering).
- **`"mayoritas"` (default v7):** semua tanggal dalam tahun itu dipakai. Untuk tiap piksel dihitung berapa persen tanggal valid yang mengklasifikasikannya terbangun; kalau **lebih dari** `MAJORITY_MIN_FRAC` (50%), piksel itu terbangun di tahun tersebut. Bangunan terbaca terbangun di hampir semua tanggal, sedangkan sawah bera atau tanah kering hanya di sebagian tanggal, jadi kesalahan sesaat tersaring. Seri dihitung nonterbangun: kalau suatu tahun hanya punya 2 tanggal, piksel harus terbangun di keduanya. Alasannya, kesalahan yang paling merusak di v6 adalah sawah bera atau tanah kering yang terbaca terbangun (luas terbangun Mekar Baru 2025 mencapai 13,5 km²), jadi aturan seri dibuat untuk menyaring kesalahan jenis itu. Tahun yang hanya punya 1 tanggal (2016) tetap memakai tanggal itu.

Setelah itu, kalau `TEMPORAL_CONSISTENCY = True`, diterapkan aturan: **piksel yang terbangun di dua tahun sebelumnya berturut-turut dianggap tetap terbangun**. Dasarnya, lahan terbangun praktis tidak kembali jadi nonterbangun dalam rentang beberapa tahun. Aturan ini hanya melihat tahun sebelumnya, tidak melihat tahun sesudahnya, jadi peta 2024 dan 2025 yang dipakai sebagai data test tidak mendapat informasi dari masa depan. Konsekuensinya perlu ditulis di metode: model diuji terhadap label yang sudah dibersihkan dengan aturan yang sama.

Kedua langkah dikerjakan di resolusi asli 10 m, baru kemudian peta di-resize ke 64×64 dengan *nearest neighbour* (label kategorikal tidak boleh diinterpolasi).

Hasilnya disimpan ke cache pickle di Drive. File `.tif` hanya dibaca sekali; kalau cache sudah ada, sel ini selesai dalam hitungan detik.

Membangun cache mayoritas membaca **semua** file `.tif` sekali (lebih lama dari v6, bisa puluhan menit). Progresnya disimpan per kecamatan, jadi kalau putus tinggal jalankan ulang.

Tabel diagnosis yang dicetak di akhir:
- **Jumlah tanggal per tahun** (mode mayoritas) atau **audit tanggal** (mode tanggal terbaik).
- **Stabilitas label**: total luas terbangun tiap tahun (resolusi asli) dan persentase piksel yang berubah 0→1 dan 1→0 dari tahun ke tahun. Ini cara cepat menilai apakah peta tahunannya masuk akal **sebelum** melatih model. Label yang sehat: luas naik pelan dan stabil, 0→1 kecil, 1→0 mendekati nol. Sebagai pembanding, di v6 perubahan 2023→2024 mencapai 13,4% piksel (9,2% jadi terbangun, 4,1% jadi nonterbangun).
- **Keutuhan resize**: kemiripan peta asli dengan peta 64×64 yang dikembalikan ke ukuran asli.

In [ ]:
def discover_kecamatan(kota_dir):
    return sorted([
        d for d in os.listdir(kota_dir) if os.path.isdir(os.path.join(kota_dir, d))
    ])


def _month_of(date_name):
    digits = re.sub(r"\D", "", date_name)   # "2019-03-26" atau "20190326" -> "20190326"
    return int(digits[4:6])


def build_annual_composite(kota, kecamatan, allowed_months=None, return_dates=False):
    """
    Tanggal-tanggal yang tidak beraturan -> 1 peta representatif per TAHUN
    (dipilih tanggal dengan piksel valid TERBANYAK di tahun itu -- proxy
    kualitas terbaik, karena piksel nodata biasanya sisa awan/gap).
    """
    kec_dir = os.path.join(RF_OUTPUT_ROOT, kota, kecamatan)
    # Folder tanggal = tanggal akuisisi (mis. 2025-09-22). Nama file di dalamnya tidak harus sama
    # dengan nama folder (mis. RF_LABEL_2025-11-01.tif), jadi file dicari dengan pola, bukan nama persis.
    date_folders = sorted([
        d for d in os.listdir(kec_dir)
        if os.path.isdir(os.path.join(kec_dir, d)) and re.match(r"^\d{4}-?\d{2}-?\d{2}", d)
    ])

    by_year = {}       # kandidat dari semua tanggal (perilaku lama)
    by_year_win = {}   # kandidat yang jatuh di allowed_months (kalau diisi)
    for date_name in date_folders:
        year = int(date_name[:4])
        tifs = sorted(glob.glob(os.path.join(kec_dir, date_name, "RF_LABEL_*.tif")))
        if not tifs:
            continue
        path = tifs[0]
        if len(tifs) > 1:
            print(f"[peringatan] {kota}/{kecamatan}/{date_name} berisi {len(tifs)} file RF_LABEL, dipakai: {os.path.basename(path)}")
        with rasterio.open(path) as src:
            arr = src.read(1)
        n_valid = int(np.sum(arr != LABEL_NODATA))
        cand = (arr, n_valid, date_name)
        if year not in by_year or n_valid > by_year[year][1]:
            by_year[year] = cand
        if allowed_months is not None:
            lo, hi = allowed_months
            if lo <= _month_of(date_name) <= hi:
                if year not in by_year_win or n_valid > by_year_win[year][1]:
                    by_year_win[year] = cand

    # kalau suatu tahun tidak punya tanggal di jendela bulan, pakai semua tanggal tahun itu
    chosen = {y: by_year_win.get(y, by_year[y]) for y in by_year}

    years_sorted = sorted(chosen.keys())
    maps = [chosen[y][0] for y in years_sorted]
    if return_dates:
        return years_sorted, maps, [chosen[y][2] for y in years_sorted]
    return years_sorted, maps


def resize_to_patch(arr, size=PATCH_SIZE):
    """Resize label biner ke ukuran seragam pakai NEAREST (kategorikal -- tidak
    boleh diinterpolasi, sama alasannya seperti diskusi resampling SCL dulu)."""
    valid = (arr != LABEL_NODATA).astype(np.float32)
    binary = np.where(arr == 1, 1.0, 0.0).astype(np.float32)

    binary_r = sk_resize(binary, (size, size), order=0, preserve_range=True, anti_aliasing=False)
    valid_r = sk_resize(valid, (size, size), order=0, preserve_range=True, anti_aliasing=False)

    out = binary_r.copy()
    out[valid_r < 0.5] = -1  # tandai nodata versi resize (dipakai untuk masking evaluasi nanti)
    return out.astype(np.float32)


def resize_fidelity(arr):
    """Seberapa mirip peta asli dengan peta yang di-resize ke 64x64 lalu dikembalikan ke ukuran asli (nearest)."""
    up = sk_resize(resize_to_patch(arr), arr.shape, order=0, preserve_range=True, anti_aliasing=False)
    valid = (arr != LABEL_NODATA)
    t = (arr == 1)[valid]
    p = (up == 1)[valid]
    oa = float(np.mean(t == p))
    iou = float((t & p).sum() / max(1, (t | p).sum()))
    return oa, iou


ANNUAL_CACHE_PATH = os.path.join(SKRIPSI_DIR, f"convlstm_annual_cache_{DATA_TAG}.pkl")


def list_rf_files(kota, kecamatan):
    """{tahun: [(nama_folder_tanggal, path_tif), ...]} untuk satu kecamatan."""
    kec_dir = os.path.join(RF_OUTPUT_ROOT, kota, kecamatan)
    out = {}
    for d in sorted(os.listdir(kec_dir)):
        if not (os.path.isdir(os.path.join(kec_dir, d)) and re.match(r"^\d{4}-?\d{2}-?\d{2}", d)):
            continue
        tifs = sorted(glob.glob(os.path.join(kec_dir, d, "RF_LABEL_*.tif")))
        if tifs:
            out.setdefault(int(d[:4]), []).append((d, tifs[0]))
    return out


def build_annual_majority(kota, kecamatan, allowed_months=None, min_frac=MAJORITY_MIN_FRAC):
    """
    1 peta per tahun dari SEMUA tanggal: piksel = 1 kalau terbangun di LEBIH DARI min_frac tanggal valid (seri = 0),
    0 kalau tidak, LABEL_NODATA kalau tidak pernah valid. Return years, maps, jumlah_tanggal_per_tahun.
    """
    files = list_rf_files(kota, kecamatan)
    years, maps, n_dates = [], [], []
    for year in sorted(files):
        cand = files[year]
        if allowed_months is not None:
            lo, hi = allowed_months
            win = [c for c in cand if lo <= _month_of(c[0]) <= hi]
            cand = win if win else cand          # tahun tanpa tanggal di jendela: pakai semua tanggal
        n_built = n_valid = None
        used = 0
        for date_name, path in cand:
            with rasterio.open(path) as src:
                arr = src.read(1)
            if n_built is None:
                n_built = np.zeros(arr.shape, np.uint16)
                n_valid = np.zeros(arr.shape, np.uint16)
            elif arr.shape != n_built.shape:
                print(f"[peringatan] {kota}/{kecamatan}/{date_name}: ukuran {arr.shape} beda, dilewati")
                continue
            valid = (arr != LABEL_NODATA)
            n_valid += valid
            n_built += valid & (arr == 1)
            used += 1
        frac = n_built / np.maximum(n_valid, 1)
        annual = (frac > min_frac).astype(np.uint8)
        annual[n_valid == 0] = LABEL_NODATA
        years.append(year)
        maps.append(annual)
        n_dates.append(used)
    return years, maps, n_dates


def apply_temporal_consistency(maps):
    """Piksel yang terbangun di 2 tahun sebelumnya (setelah dibersihkan) dipaksa tetap terbangun.
    Hanya memakai tahun sebelumnya, tidak pernah tahun sesudahnya. Nodata dibiarkan."""
    out = [m.copy() for m in maps]
    for t in range(2, len(out)):
        if out[t].shape != out[t - 1].shape or out[t].shape != out[t - 2].shape:
            continue
        keep = (out[t - 1] == 1) & (out[t - 2] == 1) & (out[t] == 0)
        out[t][keep] = 1
    return out


def load_or_build_annual_cache():
    cache = {}
    if os.path.exists(ANNUAL_CACHE_PATH):
        with open(ANNUAL_CACHE_PATH, "rb") as f:
            cache = pickle.load(f)
        print(f"Cache ditemukan: {len(cache)} kecamatan sudah tersimpan, sisanya dilanjutkan.")

    for kota in SELECTED_KOTA:
        kota_dir = os.path.join(RF_OUTPUT_ROOT, kota)
        if not os.path.isdir(kota_dir):
            print(f"[skip kota] folder belum ada: {kota_dir}")
            continue
        for kecamatan in discover_kecamatan(kota_dir):
            kec_id = f"{kota}__{kecamatan}"
            if kec_id in cache:
                continue
            if COMPOSITE_MODE == "mayoritas":
                years, maps, n_dates = build_annual_majority(kota, kecamatan, COMPOSITE_MONTHS)
                dates = [f"{n} tanggal" for n in n_dates]
            else:
                years, maps, dates = build_annual_composite(
                    kota, kecamatan, allowed_months=COMPOSITE_MONTHS, return_dates=True)
                n_dates = [1] * len(years)
            if len(years) == 0:
                print(f"{kec_id}: [dilewati] tidak ada raster RF_LABEL")
                continue
            if TEMPORAL_CONSISTENCY:
                maps = apply_temporal_consistency(maps)
            cache[kec_id] = {
                "years": years,
                "chosen_dates": dates,
                "n_dates": n_dates,
                "maps": np.stack([resize_to_patch(m) for m in maps]),   # (n_tahun, 64, 64)
                "orig_shape": tuple(maps[-1].shape),
                "orig_valid_px": [int(np.sum(m != LABEL_NODATA)) for m in maps],
                "orig_built_px": [int(np.sum(m == 1)) for m in maps],
                "resize_oa": [resize_fidelity(m)[0] for m in maps],
                "resize_iou": [resize_fidelity(m)[1] for m in maps],
            }
            with open(ANNUAL_CACHE_PATH, "wb") as f:      # simpan tiap kecamatan -> aman kalau putus
                pickle.dump(cache, f)
            print(f"{kec_id}: {len(years)} tahun {years[0]}-{years[-1]}, raster asli {cache[kec_id]['orig_shape']}")
    return cache


def label_stability(annual):
    """Luas terbangun total per tahun (resolusi asli) dan % piksel berubah antar tahun (peta 64x64)."""
    rows = []
    for t, year in enumerate(YEARS_ALL):
        row = {"tahun": year}
        for name, keys in [("total", list(annual)), ("kabupaten", [k for k in annual if k.startswith("kabupaten")])]:
            row[f"luas_{name}_km2"] = sum(annual[k]["orig_built_px"][t] for k in keys) * PIXEL_AREA_KM2
        if t > 0:
            a = np.stack([annual[k]["maps"][t - 1] for k in annual])
            b = np.stack([annual[k]["maps"][t] for k in annual])
            valid = (a != -1) & (b != -1)
            n = valid.sum()
            row["%_0ke1"] = ((a == 0) & (b == 1) & valid).sum() / n * 100
            row["%_1ke0"] = ((a == 1) & (b == 0) & valid).sum() / n * 100
        rows.append(row)
    return pd.DataFrame(rows).set_index("tahun")


def audit_chosen_dates(cache):
    rows = [(yr, dt) for d in cache.values() for yr, dt in zip(d["years"], d["chosen_dates"])]
    df = pd.DataFrame(rows, columns=["tahun", "tanggal"])
    return df.groupby("tahun")["tanggal"].agg(
        tanggal_modus=lambda s: s.value_counts().index[0],
        dipilih_oleh=lambda s: f"{s.value_counts().iloc[0]}/{len(s)} kecamatan",
        jumlah_tanggal_berbeda="nunique",
    )


CACHE = load_or_build_annual_cache()

# hanya kecamatan dengan deret tahun lengkap yang dipakai (supaya jendela waktunya sejajar)
ANNUAL = {k: v for k, v in CACHE.items() if v["years"] == YEARS_ALL}
dropped = sorted(set(CACHE) - set(ANNUAL))

# filter wilayah / kecamatan sesuai konfigurasi (form: Wilayah, Kecamatan)
ANNUAL = {k: v for k, v in ANNUAL.items() if k.split("__")[0] in SELECTED_KOTA}
if SELECTED_KECAMATAN:
    ANNUAL = {k: v for k, v in ANNUAL.items() if k.split("__")[1] in SELECTED_KECAMATAN}
KEC_IDS = sorted(ANNUAL.keys())
print(f"\nKecamatan dipakai: {len(KEC_IDS)} | dibuang karena tahun tidak lengkap: {len(dropped)} {dropped}")


if COMPOSITE_MODE == "mayoritas":
    nd = pd.DataFrame([dict(zip(v["years"], v["n_dates"])) for v in ANNUAL.values()])
    print("\nJumlah tanggal yang dipakai per tahun (min / median / maks antar kecamatan):")
    print(nd.agg(["min", "median", "max"]).T.astype(int).to_string())
else:
    print("\nAudit tanggal yang terpilih sebagai wakil tiap tahun:")
    print(audit_chosen_dates(ANNUAL).to_string())

print("\nStabilitas label (luas = resolusi asli; % = piksel valid 64x64 yang berubah dari tahun sebelumnya):")
STABILITY = label_stability(ANNUAL)
display(STABILITY.round(2))
STABILITY.to_csv(os.path.join(OUTPUT_DIR, "stabilitas_label.csv"))

fid = pd.DataFrame({
    "OA": {k: np.mean(v["resize_oa"]) for k, v in ANNUAL.items()},
    "IoU": {k: np.mean(v["resize_iou"]) for k, v in ANNUAL.items()},
})
print(f"\nKeutuhan peta asli setelah di-resize ke {PATCH_SIZE}x{PATCH_SIZE} lalu dikembalikan (antar kecamatan, rata-rata antar tahun):")
print(fid.describe().loc[["min", "50%", "max"]].round(3).to_string())

## 2. Pembagian tahun: train, validasi, test

Satu sampel = `seq_len` peta berurutan → peta tahun berikutnya. Pembagian dilakukan **menurut tahun target**, bukan acak dan bukan per kecamatan, karena yang mau diuji adalah kemampuan meramal tahun yang belum terlihat. Untuk deret waktu, Cerqueira, Torgo & Mozetič (2020, *Machine Learning* 109) menunjukkan evaluasi yang menjaga urutan waktu memberi perkiraan galat yang paling bisa dipercaya.

Dengan data 2016–2025 dan `seq_len` maksimum 4, tahun target yang bisa dipakai adalah 2020–2025. Rasio 0,5 / 0,17 / 0,33 memotongnya jadi:
- **train** 2020–2022: untuk melatih tiap kombinasi grid,
- **validasi** 2023: untuk memilih kombinasi terbaik,
- **test** 2024 dan 2025: baru dibuka setelah kombinasi dipilih. 2024 diprediksi 1 langkah dari data 2023 (meniru prediksi 2026), 2025 diprediksi 2 langkah secara rekursif (meniru prediksi 2027).

Semua kecamatan dilatih dalam **satu model bersama**, bukan satu model per kecamatan. Satu kecamatan hanya punya 10 peta, terlalu sedikit untuk melatih ConvLSTM sendiri. Di web, pilihan wilayah cukup menentukan peta mana yang dimasukkan ke model.

In [ ]:
def stack_maps(year):
    """Peta tahun tertentu untuk semua kecamatan -> (K, H, W), urutan sesuai KEC_IDS."""
    return np.stack([
        ANNUAL[k]["maps"][ANNUAL[k]["years"].index(year)] for k in KEC_IDS
    ])


def make_samples(target_years, seq_len):
    """
    Satu sample = (seq_len tahun berurutan) -> (tahun berikutnya).
    Return X (N, seq_len, H, W, 1), y (N, H, W, 1), meta [(kec_id, tahun_target), ...]
    """
    X, y, meta = [], [], []
    for kec_id in KEC_IDS:
        d = ANNUAL[kec_id]
        idx = {yr: i for i, yr in enumerate(d["years"])}
        for ty in target_years:
            in_years = list(range(ty - seq_len, ty))
            if ty not in idx or any(yy not in idx for yy in in_years):
                continue
            X.append(d["maps"][[idx[yy] for yy in in_years]])   # (seq_len, H, W)
            y.append(d["maps"][idx[ty]])                         # (H, W)
            meta.append((kec_id, ty))
    X = np.array(X, dtype=np.float32)[..., np.newaxis]
    y = np.array(y, dtype=np.float32)[..., np.newaxis]
    return X, y, meta


def split_years_by_ratio(years, train_ratio, val_ratio, test_ratio):
    """Potong daftar tahun target menurut urutan waktu. Val dan test masing-masing minimal 1 tahun kalau rasionya > 0."""
    n = len(years)
    tot = train_ratio + val_ratio + test_ratio
    n_test = max(1, round(n * test_ratio / tot)) if test_ratio > 0 else 0
    n_val = max(1, round(n * val_ratio / tot)) if val_ratio > 0 else 0
    n_train = n - n_val - n_test
    if n_train < 1:
        raise ValueError("Rasio terlalu kecil untuk train: tidak ada tahun yang tersisa.")
    return years[:n_train], years[n_train:n_train + n_val], years[n_train + n_val:]


MAX_SEQ = max(PARAM_GRID["sequence_length"])
CANDIDATE_TARGETS = list(range(YEARS_ALL[0] + MAX_SEQ, LAST_OBSERVED_YEAR + 1))     # 2020 ... 2025
TRAIN_TARGETS, VAL_TARGETS, TEST_TARGETS = split_years_by_ratio(
    CANDIDATE_TARGETS, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
ORIGIN = TEST_TARGETS[0] - 1       # tahun terakhir yang "terlihat" saat uji test (2023)
print(f"Train: {TRAIN_TARGETS} | Validasi: {VAL_TARGETS} | Test: {TEST_TARGETS} | data terakhir saat test: {ORIGIN}")

## 3. Model ConvLSTM (rumus LSTM ditulis sendiri)

**Pembagian kerja:**

| Ditulis sendiri di sel ini | Dari TensorFlow/Keras |
|---|---|
| Rumus gate dan state, Persamaan 2.39–2.44 (`ConvLSTMManual.step`) | Operasi konvolusi `tf.nn.conv2d` (tanda ∗ di rumus) |
| Perulangan tahun: membaca peta satu per satu, meneruskan H dan C (`ConvLSTMManual.call`) | Penghitungan gradien otomatis (backpropagation through time) |
| State awal H₀ = C₀ = 0 | Optimizer Adam |
| Pilihan mengeluarkan H semua tahun atau tahun terakhir saja (`return_sequences`) | `BatchNormalization`, `Conv2D` output, proses `fit` |

**Rumus yang dihitung `ConvLSTMManual.step` di setiap tahun t** (varian tanpa *peephole*, sama dengan implementasi `ConvLSTM2D` Keras):

- i_t = σ(W_xi ∗ X_t + W_hi ∗ H_{t−1} + b_i) (2.39)
- f_t = σ(W_xf ∗ X_t + W_hf ∗ H_{t−1} + b_f) (2.40)
- C̃_t = tanh(W_xc ∗ X_t + W_hc ∗ H_{t−1} + b_c) (2.41)
- C_t = f_t ⊙ C_{t−1} + i_t ⊙ C̃_t (2.42)
- o_t = σ(W_xo ∗ X_t + W_ho ∗ H_{t−1} + b_o) (2.43)
- H_t = o_t ⊙ tanh(C_t) (2.44)

Persamaan asli Shi dkk. (2015) memiliki tambahan suku *peephole* (W_ci ∘ C_{t−1}, W_cf ∘ C_{t−1}, W_co ∘ C_t). Implementasi Keras dan notebook ini tidak memakainya.

**Satu layer** = satu paket bobot (W_x, W_h, b untuk 4 gate) beserta memorinya (H, C). Layer pertama menerima peta 0/1; layer berikutnya menerima H dari layer sebelumnya. Bobot satu layer dipakai ulang untuk semua tahun dalam urutan.

**Susunan model:** `n_layers` × (`ConvLSTMManual` → `BatchNormalization`), lalu `Conv2D` 1 filter dengan sigmoid yang mengeluarkan peluang terbangun per piksel (0–1).

**Loss-nya *masked binary cross-entropy*:** piksel nodata (−1 setelah resize) tidak ikut dihitung.

Model yang disimpan (`.keras`) memakai layer `ConvLSTMManual`, jadi untuk memuatnya di aplikasi web, kelas `ConvLSTMManual` (sel di bawah) harus didefinisikan dulu sebelum `tf.keras.models.load_model(path, compile=False)`.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="sipeta")
class ConvLSTMManual(layers.Layer):
    """
    Satu layer ConvLSTM yang RUMUSNYA DITULIS SENDIRI (Persamaan 2.39-2.44, varian tanpa peephole).

    - Operasi konvolusi (tanda * di rumus) memakai tf.nn.conv2d dari TensorFlow, padding "same"
      (zero-padding seperti Shi dkk. 2015 bagian 3.1, jadi peta tetap berukuran 64x64).
    - State awal H_0 = C_0 = 0 (Shi dkk. 2015 bagian 3.1).
    - Bobot yang SAMA dipakai di semua langkah waktu; yang berubah dari tahun ke tahun hanya H dan C.
    - Nilai awal bobot mengikuti default ConvLSTM2D Keras: Glorot uniform untuk bobot input,
      orthogonal untuk bobot hidden state, bias 0 kecuali bias forget gate = 1 (unit_forget_bias).
    - dropout (opsional, default 0): seperti ConvLSTM2D Keras, sebagian nilai input X_t dinolkan
      saat latihan saja, dengan pola yang sama untuk semua tahun dalam satu urutan.
    """

    GATES = ("i", "f", "c", "o")    # input gate, forget gate, kandidat cell state, output gate

    def __init__(self, filters, kernel_size, return_sequences=False, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.kernel_size = int(kernel_size)
        self.return_sequences = bool(return_sequences)
        self.dropout = float(dropout)

    def get_config(self):           # supaya model .keras bisa disimpan dan dimuat ulang
        return {**super().get_config(), "filters": self.filters, "kernel_size": self.kernel_size,
                "return_sequences": self.return_sequences, "dropout": self.dropout}

    def build(self, input_shape):   # input_shape = (batch, waktu, tinggi, lebar, kanal)
        k, f, c_in = self.kernel_size, self.filters, int(input_shape[-1])
        # Nilai awal 4 gate dibuat sekaligus lalu dipotong per gate, supaya sebarannya sama dengan ConvLSTM2D.
        W_x_awal = tf.keras.initializers.GlorotUniform()((k, k, c_in, 4 * f))
        W_h_awal = tf.keras.initializers.Orthogonal()((k, k, f, 4 * f))
        self.W_x, self.W_h, self.b = {}, {}, {}
        for j, g in enumerate(self.GATES):
            sl = slice(j * f, (j + 1) * f)
            self.W_x[g] = self.add_weight(name=f"W_x{g}", shape=(k, k, c_in, f),
                                          initializer=lambda shape, dtype=None, v=W_x_awal[..., sl]: v)
            self.W_h[g] = self.add_weight(name=f"W_h{g}", shape=(k, k, f, f),
                                          initializer=lambda shape, dtype=None, v=W_h_awal[..., sl]: v)
            self.b[g] = self.add_weight(name=f"b_{g}", shape=(f,), initializer="ones" if g == "f" else "zeros")

    def conv(self, x, w):
        """Tanda * di rumus: konvolusi 2D dari TensorFlow (kernel tidak dibalik / cross-correlation)."""
        return tf.nn.conv2d(x, w, strides=1, padding="SAME")

    def step(self, X_t, H_prev, C_prev, masks=None):
        """Satu langkah waktu: Persamaan 2.39-2.44."""
        x = {g: (X_t * masks[g] if masks else X_t) for g in self.GATES}
        i_t = tf.sigmoid(self.conv(x["i"], self.W_x["i"]) + self.conv(H_prev, self.W_h["i"]) + self.b["i"])      # (2.39)
        f_t = tf.sigmoid(self.conv(x["f"], self.W_x["f"]) + self.conv(H_prev, self.W_h["f"]) + self.b["f"])      # (2.40)
        C_tilde = tf.tanh(self.conv(x["c"], self.W_x["c"]) + self.conv(H_prev, self.W_h["c"]) + self.b["c"])     # (2.41)
        C_t = f_t * C_prev + i_t * C_tilde                                                                       # (2.42)
        o_t = tf.sigmoid(self.conv(x["o"], self.W_x["o"]) + self.conv(H_prev, self.W_h["o"]) + self.b["o"])      # (2.43)
        H_t = o_t * tf.tanh(C_t)                                                                                 # (2.44)
        return H_t, C_t

    def call(self, X, training=None):
        shape = tf.shape(X)
        H = tf.zeros((shape[0], shape[2], shape[3], self.filters), dtype=X.dtype)   # H_0 = 0
        C = tf.zeros_like(H)                                                         # C_0 = 0
        masks = None
        if training and self.dropout > 0:
            ones = tf.ones_like(X[:, 0])
            masks = {g: tf.nn.dropout(ones, rate=self.dropout) for g in self.GATES}
        outputs = []
        for t in range(X.shape[1]):              # baca tahun satu per satu, dari yang tertua
            H, C = self.step(X[:, t], H, C, masks)
            outputs.append(H)
        # return_sequences=True: keluarkan H semua tahun (untuk layer ConvLSTM berikutnya)
        # return_sequences=False: keluarkan H tahun terakhir saja (untuk Conv2D output)
        return tf.stack(outputs, axis=1) if self.return_sequences else H


def masked_binary_crossentropy(y_true, y_pred):
    """
    BCE yang benar-benar mengabaikan piksel nodata (-1), bukan cuma
    dihitung normal lalu 'untung-untungan'. Piksel -1 di-nolkan bobotnya
    total, tidak ikut menyumbang ke loss sama sekali.
    """
    y_true = tf.cast(y_true, tf.float32)
    mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
    y_true_safe = tf.where(tf.equal(y_true, -1.0), tf.zeros_like(y_true), y_true)

    bce = tf.keras.losses.binary_crossentropy(y_true_safe, y_pred)  # shape (batch,H,W)
    mask_2d = tf.squeeze(mask, axis=-1)

    return tf.reduce_sum(bce * mask_2d) / (tf.reduce_sum(mask_2d) + 1e-8)


def masked_accuracy(y_true, y_pred):
    """Accuracy yang juga mengabaikan piksel nodata -- biar log training
    selama fit() jujur menggambarkan performa asli, bukan tersamarkan."""
    y_true = tf.cast(y_true, tf.float32)
    mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
    y_pred_bin = tf.cast(y_pred > 0.5, tf.float32)
    y_true_safe = tf.where(tf.equal(y_true, -1.0), tf.zeros_like(y_true), y_true)

    correct = tf.cast(tf.equal(y_true_safe, y_pred_bin), tf.float32) * mask
    return tf.reduce_sum(correct) / (tf.reduce_sum(mask) + 1e-8)


OPTIMIZERS = {
    "adam": lambda lr: tf.keras.optimizers.Adam(learning_rate=lr),
    "adamw": lambda lr: tf.keras.optimizers.AdamW(learning_rate=lr),
    "nadam": lambda lr: tf.keras.optimizers.Nadam(learning_rate=lr),
    "rmsprop": lambda lr: tf.keras.optimizers.RMSprop(learning_rate=lr),
    "sgd": lambda lr: tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
}


def build_convlstm_model(seq_len, patch_size, filters=40, kernel_size=(3, 3), learning_rate=0.001,
                         n_layers=2, dropout=0.0, optimizer="adam"):
    """
    n_layers ConvLSTMManual (tiap layer diikuti BatchNormalization), lalu Conv2D sigmoid -> peluang terbangun.
    Layer ConvLSTM terakhir hanya mengeluarkan H tahun terakhir (return_sequences=False).
    """
    k = int(kernel_size[0]) if isinstance(kernel_size, (tuple, list)) else int(kernel_size)
    inputs = layers.Input(shape=(seq_len, patch_size, patch_size, 1))

    x = inputs
    for i in range(n_layers):
        last = (i == n_layers - 1)
        x = ConvLSTMManual(filters, k, return_sequences=not last, dropout=dropout, name=f"convlstm_{i + 1}")(x)
        x = layers.BatchNormalization(name=f"batchnorm_{i + 1}")(x)

    outputs = layers.Conv2D(filters=1, kernel_size=(k, k), padding="same", activation="sigmoid",
                            name="output_peluang")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=OPTIMIZERS[optimizer.lower()](learning_rate),
        loss=masked_binary_crossentropy,
        metrics=[masked_accuracy]
    )
    return model


### 3b. Pembuktian: rumus manual = `ConvLSTM2D` Keras

Sel ini untuk lampiran skripsi atau untuk menjawab pertanyaan "dari mana tahu rumusnya benar?".

- **(a) Konvolusi.** Satu jendela 3×3 dihitung manual (kalikan per posisi lalu jumlahkan) dan dengan `tf.nn.conv2d`. Hasilnya harus sama, yaitu 0,9. TensorFlow tidak membalik kernel (secara teknis *cross-correlation*), jadi contoh perhitungan manual di skripsi juga tidak membalik kernel.
- **(b) ConvLSTM.** Satu layer `ConvLSTM2D` Keras dan satu layer `ConvLSTMManual` dibuat, bobot Keras disalin ke layer manual, lalu keduanya diberi input yang sama. Selisih maksimum harus mendekati nol (sekitar 10⁻⁸, sisa pembulatan angka desimal). Kalau tidak, sel berhenti dengan error.

Di akhir ditampilkan susunan model contoh beserta jumlah parameternya.

In [ ]:
# (a) Konvolusi: hitung manual satu jendela 3x3, bandingkan dengan tf.nn.conv2d
X_demo = np.array([[0, 1, 1], [0, 1, 0], [1, 1, 0]], np.float32)
W_demo = np.array([[0.2, -0.1, 0.3], [0.0, 0.5, 0.1], [-0.2, 0.4, 0.0]], np.float32)
hasil_manual = float(np.sum(X_demo * W_demo))      # kalikan per posisi lalu jumlahkan (kernel tidak dibalik)
hasil_tf = float(tf.nn.conv2d(X_demo[None, :, :, None], W_demo[:, :, None, None], strides=1, padding="VALID")[0, 0, 0, 0])
print(f"(a) Konvolusi satu jendela 3x3: manual = {hasil_manual:.4f} | tf.nn.conv2d = {hasil_tf:.4f}")

# (b) ConvLSTMManual vs ConvLSTM2D Keras dengan bobot yang sama
rng_cek = np.random.default_rng(RANDOM_STATE)
X_cek = tf.constant(rng_cek.integers(0, 2, (2, 3, PATCH_SIZE, PATCH_SIZE, 1)).astype("float32"))
rows = []
for n_f, k in [(8, 3), (8, 5)]:
    for rs in (True, False):
        ref = layers.ConvLSTM2D(n_f, (k, k), padding="same", return_sequences=rs)
        y_ref = ref(X_cek)
        man = ConvLSTMManual(n_f, k, return_sequences=rs)
        man(X_cek)                                           # membuat bobot
        W_x_all, W_h_all, b_all = ref.get_weights()          # Keras menyimpan 4 gate berurutan: i, f, c, o
        for j, g in enumerate(man.GATES):
            sl = slice(j * n_f, (j + 1) * n_f)
            man.W_x[g].assign(W_x_all[..., sl])
            man.W_h[g].assign(W_h_all[..., sl])
            man.b[g].assign(b_all[sl])
        diff = float(np.max(np.abs(y_ref.numpy() - man(X_cek).numpy())))
        rows.append({"filter": n_f, "kernel": f"{k}x{k}", "return_sequences": rs,
                     "bentuk keluaran": tuple(y_ref.shape), "selisih maksimum": diff})
cek = pd.DataFrame(rows)
print("(b) ConvLSTMManual vs ConvLSTM2D Keras (bobot disalin, input sama):")
display(cek)
assert cek["selisih maksimum"].max() < 1e-4, "Rumus manual TIDAK sama dengan Keras"
print("Lulus: keluaran rumus manual sama dengan ConvLSTM2D Keras (selisih hanya pembulatan angka desimal).")

contoh = build_convlstm_model(4, PATCH_SIZE, filters=40, kernel_size=(5, 5), n_layers=2)
print("\nSusunan model contoh (40 filter, kernel 5x5, 4 tahun, 2 layer):")
contoh.summary()
del contoh


## 4. Fungsi evaluasi

**Metrik peta (per piksel, piksel nodata dibuang).** Dari confusion matrix (TP, FP, FN, TN) dihitung akurasi, presisi, recall, F1, dan IoU kelas terbangun. Masalahnya, sebagian besar piksel tidak berubah dari tahun ke tahun, sehingga baseline persistensi (salin peta tahun lalu) pun sudah mendapat akurasi dan IoU tinggi. Karena itu ditambah **FoM** (*Figure of Merit*, Pontius dkk., 2008, *Annals of Regional Science* 42:11–37) yang hanya menilai piksel yang berubah dari nonterbangun ke terbangun:

FoM = hits / (hits + misses + false alarms)

FoM persistensi selalu 0, jadi FoM > 0 berarti model menangkap perubahan yang tidak bisa ditebak dengan menyalin peta.

**Metrik luas (per kecamatan, km²).** Luas terbangun prediksi dibandingkan dengan luas aktual di tiap kecamatan, lalu dihitung MAE, RMSE, MAPE, R², dan bias total (selisih total luas seluruh wilayah, dalam persen; negatif = model meremehkan luas). Inilah tempat MAE/RMSE/MAPE/R² bermakna. Pada peta biner per piksel, MAE sama dengan 1 − akurasi dan RMSE sama dengan √(1 − akurasi), jadi tidak menambah informasi dan tidak dipakai di level piksel. R² luas biasanya tinggi untuk model maupun persistensi karena luas kecamatan memang sangat berbeda-beda, jadi bacanya selalu dibandingkan dengan R² persistensi.

Luas dihitung dengan skala yang benar: 1 piksel 64×64 mewakili (tinggi × lebar raster asli) / 64² piksel 10 m.

**Prediksi rekursif.** Untuk langkah ke-2 (2025 di test, 2027 di prediksi), peta biner hasil langkah ke-1 dimasukkan kembali sebagai input.

In [ ]:
def _div(a, b):
    return float(a) / float(b) if b > 0 else float("nan")


def _spatial_scores(t, p):
    tp = int(np.sum(p & t)); fp = int(np.sum(p & ~t))
    fn = int(np.sum(~p & t)); tn = int(np.sum(~p & ~t))
    return {
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "oa": _div(tp + tn, tp + fp + fn + tn),
        "precision": _div(tp, tp + fp),
        "recall": _div(tp, tp + fn),
        "f1": _div(2 * tp, 2 * tp + fp + fn),
        "iou": _div(tp, tp + fp + fn),
    }


def evaluate_predictions(before, y_true, y_prob, threshold=THRESHOLD):
    """
    before : (N,H,W) peta kondisi awal (tahun terakhir yang teramati; -1 = nodata)
    y_true : (N,H,W) peta aktual tahun target (-1 = nodata)
    y_prob : (N,H,W) probabilitas terbangun dari model
    Rumus IoU & FoM sama dengan kode lama; ditambah baseline persistensi.
    """
    y_pred = (y_prob > threshold)
    valid = (y_true != -1)

    t = (y_true == 1)[valid]
    m = _spatial_scores(t, y_pred[valid])
    b = _spatial_scores(t, (before == 1)[valid])      # baseline: tidak ada perubahan

    valid_fom = (before != -1) & valid
    hits = int(np.sum((before == 0) & (y_true == 1) & y_pred & valid_fom))
    misses = int(np.sum((before == 0) & (y_true == 1) & ~y_pred & valid_fom))
    false_alarms = int(np.sum((before == 0) & (y_true == 0) & y_pred & valid_fom))
    fom = _div(hits, hits + misses + false_alarms)

    return {
        "oa": m["oa"], "precision": m["precision"], "recall": m["recall"],
        "f1": m["f1"], "iou": m["iou"],
        "hits": hits, "misses": misses, "false_alarms": false_alarms, "fom": fom,
        "persist_oa": b["oa"], "persist_f1": b["f1"], "persist_iou": b["iou"],
        "tp": m["tp"], "fp": m["fp"], "fn": m["fn"], "tn": m["tn"],
    }


def km2_per_resized_pixel(kec_id):
    h, w = ANNUAL[kec_id]["orig_shape"]
    return h * w * PIXEL_AREA_KM2 / (PATCH_SIZE * PATCH_SIZE)


def area_errors(before, y_true, y_prob, threshold=THRESHOLD):
    """MAE, RMSE, MAPE, R2, dan bias total untuk luas terbangun per kecamatan (km2), model dan persistensi."""
    act, pred, pers = [], [], []
    for i, kec_id in enumerate(KEC_IDS):
        s = km2_per_resized_pixel(kec_id)
        valid = (y_true[i] != -1)
        act.append(np.sum((y_true[i] == 1) & valid) * s)
        pred.append(np.sum((y_prob[i] > threshold) & valid) * s)
        pers.append(np.sum((before[i] == 1) & valid) * s)
    act, pred, pers = map(np.asarray, (act, pred, pers))

    def _scores(p):
        err = p - act
        pos = act > 0
        ss_tot = np.sum((act - act.mean()) ** 2)
        return {
            "MAE_km2": float(np.mean(np.abs(err))),
            "RMSE_km2": float(np.sqrt(np.mean(err ** 2))),
            "MAPE_%": float(np.mean(np.abs(err[pos]) / act[pos]) * 100),
            "R2": float(1 - np.sum(err ** 2) / ss_tot) if ss_tot > 0 else float("nan"),
            "bias_total_%": float(err.sum() / act.sum() * 100),
        }

    out = _scores(pred)
    out.update({f"persist_{k}": v for k, v in _scores(pers).items()})
    return out


def evaluate_all(before, y_true, y_prob):
    """Semua metrik peta + luas dalam satu dict (dipakai di grid search dan di test)."""
    m = evaluate_predictions(before, y_true, y_prob)
    m["akurasi"] = m.pop("oa")
    m["persist_akurasi"] = m.pop("persist_oa")
    valid = (y_true != -1)
    b = _spatial_scores((y_true == 1)[valid], (before == 1)[valid])
    m["persist_precision"], m["persist_recall"] = b["precision"], b["recall"]
    m["persist_fom"] = 0.0
    m.update(area_errors(before, y_true, y_prob))
    return m


def forecast_recursive(model, last_obs_year, seq_len, n_steps, threshold=THRESHOLD):
    """
    Prediksi n_steps tahun ke depan untuk SEMUA kecamatan sekaligus.
    Langkah ke-2 dst. memakai peta biner hasil prediksi langkah sebelumnya sebagai input.
    Return: list probabilitas (K,H,W) per langkah, list peta biner per langkah (-1 = nodata).
    """
    window = np.stack(
        [stack_maps(y) for y in range(last_obs_year - seq_len + 1, last_obs_year + 1)], axis=1
    )[..., np.newaxis]                                         # (K, seq_len, H, W, 1)
    nodata = (window[:, -1, :, :, 0] == -1)

    probs, bins = [], []
    for _ in range(n_steps):
        prob = model.predict(window, batch_size=BATCH_SIZE, verbose=0)[..., 0]   # (K,H,W)
        binary = (prob > threshold).astype(np.float32)
        binary[nodata] = -1.0
        probs.append(prob)
        bins.append(binary)
        window = np.concatenate([window[:, 1:], binary[:, np.newaxis, :, :, np.newaxis]], axis=1)
    return probs, bins


def confusion_from_maps(before, y_true, y_pred_bin, only_unbuilt_before=False):
    """Matriks 2x2 tingkat piksel [[TN, FP], [FN, TP]] (baris = aktual, kolom = prediksi).
    Hanya piksel valid (bukan nodata). only_unbuilt_before=True membatasi ke piksel yang tahun
    awalnya NONterbangun, yaitu piksel tempat perubahan 0->1 bisa terjadi (dasar FoM)."""
    valid = (y_true != -1) & (before != -1)
    if only_unbuilt_before:
        valid &= (before == 0)
    t = (y_true == 1)[valid]
    p = (y_pred_bin == 1)[valid]
    return np.array([[int((~t & ~p).sum()), int((~t & p).sum())],
                     [int((t & ~p).sum()), int((t & p).sum())]])


def draw_confusion(ax, cm, title, labels):
    pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, f"{cm[r, c]:,}\n({pct[r, c]:.1f}%)", ha="center", va="center", fontsize=10,
                    color="white" if pct[r, c] > 55 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(labels[1], fontsize=9); ax.set_yticklabels(labels[0], fontsize=9)
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
    ax.set_title(title, fontsize=10)

## 5. Grid search

Tiap kombinasi dilatih dengan target train (2020–2022), *early stopping* memantau `val_loss` di tahun validasi (2023), lalu model dengan bobot terbaik dievaluasi di tahun validasi memakai semua metrik di bagian 4. Satu kombinasi = satu baris di `GRID_LOG_PATH`.

**Bisa dilanjut kalau Colab putus.** Tiap kombinasi yang selesai langsung ditulis ke CSV di Drive. Saat sel ini dijalankan ulang, kombinasi yang sudah ada di log dilewati. Kalau putus di tengah satu kombinasi, `BackupAndRestore` melanjutkan dari epoch terakhir. Setelah putus, jalankan ulang semua sel dari atas: sel data memakai cache, jadi cepat.

**Ukur waktu dulu.** Sebelum menjalankan seluruh grid, ganti sementara baris terakhir sel ini jadi `grid_log = run_grid(GRID[:2])`. Lihat kolom `menit`, kalikan dengan jumlah kombinasi (108) untuk perkiraan kasar. Kombinasi 1 layer lebih cepat dan 3 layer lebih lambat dari rata-rata, lalu kembalikan baris terakhir seperti semula. Dua kombinasi tadi tetap tersimpan, tidak terbuang.

Kalau `MANUAL_CONFIG` diisi, sel ini tidak melatih apa pun.

In [ ]:
BACKUP_ROOT = os.path.join(OUTPUT_DIR, "_backup_training")
SESSION_START = time.time()
GRID_KEYS = ["filters", "kernel_size", "learning_rate", "sequence_length", "n_layers", "dropout", "optimizer"]
GRID = [dict(zip(GRID_KEYS, v)) for v in itertools.product(*[PARAM_GRID[k] for k in GRID_KEYS])]


def cfg_tag(c):
    return (f"f{c['filters']}_k{c['kernel_size']}_lr{c['learning_rate']}_L{c['sequence_length']}"
            f"_nl{c['n_layers']}_do{c['dropout']}_{c['optimizer']}")


def build_from_cfg(c):
    k = int(c["kernel_size"])
    return build_convlstm_model(int(c["sequence_length"]), PATCH_SIZE, int(c["filters"]), (k, k),
                                float(c["learning_rate"]), n_layers=int(c["n_layers"]),
                                dropout=float(c["dropout"]), optimizer=str(c["optimizer"]))


def train_and_validate(c):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    L = c["sequence_length"]
    X_tr, y_tr, _ = make_samples(TRAIN_TARGETS, L)
    X_va, y_va, _ = make_samples(VAL_TARGETS, L)

    model = build_from_cfg(c)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True,
                                         start_from_epoch=START_FROM_EPOCH),
        tf.keras.callbacks.BackupAndRestore(os.path.join(BACKUP_ROOT, cfg_tag(c))),
    ]
    t0 = time.time()
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=MAX_EPOCHS,
                     batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
    prob = model.predict(X_va, batch_size=BATCH_SIZE, verbose=0)

    m = evaluate_all(X_va[:, -1, :, :, 0], y_va[..., 0], prob[..., 0])
    best_i = int(np.argmin(hist.history["val_loss"]))
    return {"key": cfg_tag(c), **c,
            "best_epoch": best_i + 1, "epoch_jalan": len(hist.epoch),
            "val_loss": float(hist.history["val_loss"][best_i]),
            "menit": round((time.time() - t0) / 60, 2), "n_params": int(model.count_params()), **m}


def run_grid(configs):
    log = pd.read_csv(GRID_LOG_PATH) if os.path.exists(GRID_LOG_PATH) else pd.DataFrame()
    done = set(log["key"]) if len(log) else set()
    todo = [c for c in configs if cfg_tag(c) not in done]
    print(f"{len(configs) - len(todo)} kombinasi sudah ada di log, {len(todo)} tersisa")
    for n, c in enumerate(todo, start=1):
        if MAX_MINUTES_PER_SESSION and (time.time() - SESSION_START) / 60 > MAX_MINUTES_PER_SESSION:
            print("Batas waktu sesi tercapai. Jalankan sel ini lagi untuk melanjutkan.")
            break
        r = train_and_validate(c)
        log = pd.concat([log, pd.DataFrame([r])], ignore_index=True)
        log.to_csv(GRID_LOG_PATH, index=False)
        print(f"[{n}/{len(todo)}] {r['key']}: FoM {r['fom']:.4f} | IoU {r['iou']:.4f} (persistensi {r['persist_iou']:.4f}) "
              f"| F1 {r['f1']:.4f} | MAE luas {r['MAE_km2']:.2f} km2 | epoch terbaik {r['best_epoch']} | {r['menit']} menit")
    return log


print(f"Total kombinasi: {len(GRID)}")
grid_log = run_grid(GRID) if MANUAL_CONFIG is None else None

## 6. Hasil grid search dan kombinasi terbaik

Kombinasi terbaik = **FoM tertinggi di tahun validasi**, IoU sebagai pemecah seri. FoM dipakai karena yang mau diramal adalah *perubahan*; akurasi dan IoU bisa tinggi hanya karena sebagian besar piksel tidak berubah.

Tabel menampilkan 10 kombinasi teratas dengan semua metrik. Kolom `persist_...` adalah baseline persistensi, sama untuk semua kombinasi dengan `sequence_length` yang sama. Log lengkap ada di `GRID_LOG_PATH`.

Jumlah epoch untuk pelatihan ulang di langkah berikutnya diambil dari `best_epoch` kombinasi terpilih.

In [ ]:
SHOW = ["key", "fom", "iou", "persist_iou", "akurasi", "persist_akurasi", "precision", "recall", "f1",
        "tp", "fp", "fn", "tn", "MAE_km2", "RMSE_km2", "MAPE_%", "R2", "persist_MAE_km2", "persist_R2",
        "bias_total_%", "best_epoch", "menit"]

if MANUAL_CONFIG is not None:
    BEST = {k: MANUAL_CONFIG[k] for k in GRID_KEYS}
    EPOCHS_REFIT = int(MANUAL_CONFIG.get("epochs", 30))
    print("Memakai MANUAL_CONFIG (grid dilewati).")
else:
    grid_log = pd.read_csv(GRID_LOG_PATH)
    grid_log = grid_log[grid_log["key"].isin([cfg_tag(c) for c in GRID])]
    if len(grid_log) < len(GRID):
        raise RuntimeError(f"Grid baru selesai {len(grid_log)}/{len(GRID)}. Jalankan sel grid search sampai tuntas.")
    ranked = grid_log.sort_values(["fom", "iou"], ascending=False).reset_index(drop=True)
    print(f"10 kombinasi teratas (validasi tahun {VAL_TARGETS}):")
    display(ranked[SHOW].head(10).round(4))

    print("\nKombinasi terbaik untuk tiap jumlah layer ConvLSTM (validasi):")
    per_layer = ranked.groupby("n_layers", sort=True).head(1).sort_values("n_layers")
    display(per_layer[["n_layers", "key", "akurasi", "precision", "recall", "f1", "iou", "fom",
                       "n_params", "best_epoch", "menit"]].round(4).set_index("n_layers"))
    print("Rata-rata semua kombinasi per jumlah layer:")
    display(ranked.groupby("n_layers")[["akurasi", "precision", "recall", "f1", "iou", "fom", "menit"]].mean().round(4))

    best_row = ranked.iloc[0]
    BEST = {k: best_row[k] for k in GRID_KEYS}
    BEST = {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v)
            for k, v in BEST.items()}
    EPOCHS_REFIT = max(5, int(best_row["best_epoch"]))
    if best_row["best_epoch"] >= MAX_EPOCHS:
        print(f"[perhatian] epoch terbaik = batas {MAX_EPOCHS}: model masih membaik saat dihentikan.")
    if best_row["iou"] <= best_row["persist_iou"]:
        print("[perhatian] IoU model terbaik tidak lebih tinggi dari baseline persistensi di tahun validasi.")

print(f"\nKOMBINASI TERPILIH: {BEST}\nEpoch pelatihan ulang: {EPOCHS_REFIT}")
with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_best_config.json"), "w") as f:
    json.dump({**BEST, "epochs": EPOCHS_REFIT, "batch_size": BATCH_SIZE, "threshold": THRESHOLD,
               "train": TRAIN_TARGETS, "val": VAL_TARGETS, "test": TEST_TARGETS}, f, indent=2)

## 7. Uji model terbaik di data test (2024 dan 2025)

Ini angka evaluasi yang dilaporkan di skripsi. Model dengan konfigurasi terpilih dilatih ulang memakai semua target sampai 2023 (train + validasi), dengan jumlah epoch tetap `EPOCHS_REFIT`. Lalu, seolah-olah data berhenti di 2023:
- **2024** diprediksi 1 langkah, setara dengan prediksi 2026 nanti;
- **2025** diprediksi 2 langkah secara rekursif, setara dengan prediksi 2027 nanti.

Hasilnya dibandingkan dengan peta aktual dan dengan baseline persistensi (peta 2023 disalin). Untuk tiap tahun ditampilkan tabel metrik dan tiga confusion matrix: (1) model, (2) persistensi, (3) model hanya pada piksel yang di 2023 masih nonterbangun, yaitu tempat perubahan bisa terjadi (dasar FoM). Persen di dalam kotak dihitung per baris, jadi kotak kanan bawah adalah recall.

Cara membaca confusion matrix persistensi: kotak FN = piksel yang benar-benar berubah 0→1, kotak FP = piksel yang berubah 1→0. Lahan terbangun jarang kembali jadi nonterbangun, jadi kalau FP besar, sebagian perubahan di peta tahunan kemungkinan berasal dari ketidakstabilan label (musim, awan, kesalahan klasifikasi), bukan dari perubahan lahan sungguhan.

Model disimpan ke Drive. Kalau sel ini dijalankan ulang dengan konfigurasi dan epoch yang sama, model dimuat dari file (tidak dilatih ulang).

In [ ]:
BEST_TAG = cfg_tag(BEST)


def fit_and_save(cfg, target_years, epochs, path):
    """Latih dengan semua target yang diberikan (tanpa validasi, epoch tetap). Kalau file sudah ada: muat."""
    if os.path.exists(path):
        print(f"Model dimuat dari: {path}")
        return tf.keras.models.load_model(path, compile=False)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    X, y, _ = make_samples(target_years, cfg["sequence_length"])
    print(f"Melatih {len(X)} sequence, {epochs} epoch ...")
    model = build_from_cfg(cfg)
    model.fit(X, y, epochs=epochs, batch_size=BATCH_SIZE, verbose=0)
    model.save(path)
    return model


L = BEST["sequence_length"]
test_model = fit_and_save(BEST, list(range(YEARS_ALL[0] + L, ORIGIN + 1)), EPOCHS_REFIT,
                          os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_test_{BEST_TAG}_e{EPOCHS_REFIT}.keras"))
test_probs, test_bins = forecast_recursive(test_model, ORIGIN, L, len(TEST_TARGETS))
before = stack_maps(ORIGIN)

LBL_FULL = (["Nonterbangun (0)", "Terbangun (1)"], ["Nonterbangun (0)", "Terbangun (1)"])
LBL_CHG = (["Tetap nonterbangun", "Jadi terbangun (0→1)"], ["Prediksi tetap", "Prediksi jadi terbangun"])
METRIK = ["akurasi", "precision", "recall", "f1", "iou", "fom", "MAE_km2", "RMSE_km2", "MAPE_%", "R2", "bias_total_%"]

test_rows = []
for step, ty in enumerate(TEST_TARGETS):
    y_true = stack_maps(ty)
    m = evaluate_all(before, y_true, test_probs[step])
    test_rows.append({"tahun": ty, "langkah": step + 1, **m})

    tabel = pd.DataFrame({"model": [m[k] for k in METRIK], "persistensi": [m[f"persist_{k}"] for k in METRIK]},
                         index=METRIK)
    print(f"\n=== Test {ty} ({step + 1} langkah dari {ORIGIN}) ===")
    display(tabel.round(4))

    pred_bin = test_probs[step] > THRESHOLD
    cms = [confusion_from_maps(before, y_true, pred_bin),
           confusion_from_maps(before, y_true, before == 1),
           confusion_from_maps(before, y_true, pred_bin, only_unbuilt_before=True)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    draw_confusion(axes[0], cms[0], f"Model, {ty}", LBL_FULL)
    draw_confusion(axes[1], cms[1], f"Persistensi (peta {ORIGIN} disalin), {ty}", LBL_FULL)
    draw_confusion(axes[2], cms[2], f"Model, piksel nonterbangun di {ORIGIN}", LBL_CHG)
    fig.suptitle(f"Confusion matrix test {ty}", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_confusion_test_{ty}.png"), dpi=150, bbox_inches="tight")
    plt.show()

test_df = pd.DataFrame(test_rows)
test_df.to_csv(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_evaluasi_test.csv"), index=False)

## 8. Model final dan prediksi 2026–2027

Model dilatih ulang dengan **semua** data sampai 2025 memakai konfigurasi dan epoch yang sama, lalu memprediksi 2026 (1 langkah) dan 2027 (2 langkah, rekursif). Tahun 2026 dan 2027 belum punya data aktual, jadi tidak ada akurasi untuk keduanya. Seberapa bisa dipercaya prediksinya dinilai dari hasil test di langkah 7.

File yang disimpan di `convlstm_output`:
- `..._final_....keras`: model untuk dipakai di web (definisikan dulu kelas `ConvLSTMManual` dari bagian 3, lalu muat dengan `tf.keras.models.load_model(path, compile=False)`);
- `..._prediksi_luas_kecamatan.csv`: luas terbangun 2025 dan prediksi 2026–2027 per kecamatan;
- `..._prediksi_peta_2026_2027.npz`: peta probabilitas dan peta biner 64×64 per kecamatan.

Luas prediksi dibandingkan dengan luas 2025 **versi resize** (bukan resolusi asli) supaya sama-sama dihitung dari peta 64×64. Kolom `bias_resize_%` menunjukkan selisih versi resize terhadap luas resolusi asli.

In [ ]:
final_model = fit_and_save(BEST, list(range(YEARS_ALL[0] + L, LAST_OBSERVED_YEAR + 1)), EPOCHS_REFIT,
                           os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_final_{BEST_TAG}_e{EPOCHS_REFIT}.keras"))
fc_probs, fc_bins = forecast_recursive(final_model, LAST_OBSERVED_YEAR, L, N_FORECAST)
FORECAST_YEARS = [LAST_OBSERVED_YEAR + 1 + i for i in range(N_FORECAST)]

last_map = stack_maps(LAST_OBSERVED_YEAR)
rows = []
for i, kec_id in enumerate(KEC_IDS):
    d, s = ANNUAL[kec_id], km2_per_resized_pixel(kec_id)
    asli = d["orig_built_px"][d["years"].index(LAST_OBSERVED_YEAR)] * PIXEL_AREA_KM2
    resize = float(np.sum(last_map[i] == 1)) * s
    kota, kec = kec_id.split("__")
    row = {"kota": kota, "kecamatan": kec,
           f"luas_{LAST_OBSERVED_YEAR}_resolusi_asli_km2": asli,
           f"luas_{LAST_OBSERVED_YEAR}_resize_km2": resize,
           "bias_resize_%": (resize - asli) / asli * 100 if asli > 0 else np.nan}
    for step, fy in enumerate(FORECAST_YEARS):
        row[f"prediksi_{fy}_km2"] = float(np.sum(fc_bins[step][i] == 1)) * s
    rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df.to_csv(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_prediksi_luas_kecamatan.csv"), index=False)
np.savez_compressed(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_prediksi_peta_2026_2027.npz"),
                    kec_ids=np.array(KEC_IDS),
                    **{f"prob_{fy}": fc_probs[i] for i, fy in enumerate(FORECAST_YEARS)},
                    **{f"biner_{fy}": fc_bins[i] for i, fy in enumerate(FORECAST_YEARS)})

tot = pred_df[f"luas_{LAST_OBSERVED_YEAR}_resize_km2"].sum()
print(f"Total luas terbangun {LAST_OBSERVED_YEAR} (resize): {tot:,.1f} km2")
for fy in FORECAST_YEARS:
    t = pred_df[f"prediksi_{fy}_km2"].sum()
    print(f"Prediksi {fy}: {t:,.1f} km2 ({(t - tot) / tot * 100:+.2f}% dari {LAST_OBSERVED_YEAR})")
display(pred_df.round(3))

## 9. Peta biner per kecamatan

Secara default dipilih dua kecamatan: yang perubahan nyata 0→1-nya paling banyak di tahun test pertama (paling informatif) dan satu kecamatan di posisi tengah. Untuk memilih sendiri, isi `PLOT_KECS`, misalnya `["kabupaten_tangerang__sukadiri"]` (daftar ada di `KEC_IDS`).

Tiap kecamatan mendapat dua gambar:
- **Test 2024**: aktual 2023, aktual 2024, prediksi 2024, dan peta galat (hijau = perubahan tertangkap, oranye = terlewat, merah = salah alarm), plus metrik dan confusion matrix kecamatan itu.
- **Prediksi**: aktual 2025, prediksi 2026, prediksi 2027, dan peta perubahan 2025 → 2026.

Gambar **test 2024** memakai peta 64×64 (ukuran yang dipakai model).

Gambar **prediksi 2025–2027** digambar di resolusi asli 10 m dengan gaya yang sama seperti plot klasifikasi RF (hijau = non-terbangun, merah = terbangun, luas km² di legenda). Caranya: peta aktual 2025 dibuat ulang di resolusi 10 m dengan langkah yang sama seperti cache (komposit mayoritas + konsistensi waktu, tanpa resize). Lalu perubahan yang diprediksi ConvLSTM ditempelkan di atasnya: sel model yang menurut prediksi menjadi terbangun diisi terbangun, sel yang menurut prediksi hilang diisi non-terbangun, dan sel yang tidak berubah tetap memakai detail asli 2025.

Artinya, detail halus di peta 2026–2027 berasal dari peta aktual 2025. Perubahannya sendiri tetap berukuran satu sel model (ukurannya ditulis di judul panel perubahan, sekitar 100 m untuk kecamatan sedang), karena ConvLSTM memang bekerja di 64×64. Luas 10 m di bawah gambar bisa sedikit berbeda dari tabel `pred_df` yang dihitung dari versi 64×64.

In [ ]:
PLOT_KECS = None

CMAP_BIN = ListedColormap(["#e6e6e6", "#1f3b73"])
CMAP_ERR = ListedColormap(["#e6e6e6", "#2e9e4f", "#f0a030", "#d6372c"])
ty = TEST_TARGETS[0]
M_BEFORE, M_TEST, M_LAST = stack_maps(ORIGIN), stack_maps(ty), stack_maps(LAST_OBSERVED_YEAR)

if PLOT_KECS is None:
    n_change = [int(((M_BEFORE[j] == 0) & (M_TEST[j] == 1)).sum()) for j in range(len(KEC_IDS))]
    order = np.argsort(n_change)[::-1]
    PLOT_KECS = list(dict.fromkeys([KEC_IDS[order[0]], KEC_IDS[order[len(order) // 2]]]))


def show_bin(ax, m, title):
    ax.imshow(np.ma.masked_where(m == -1, m), cmap=CMAP_BIN, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def legend_below(ax, items):
    ax.legend(handles=[Patch(color=c, label=l) for c, l in items], loc="upper center",
              bbox_to_anchor=(0.5, -0.02), ncol=len(items), fontsize=8, frameon=False)


# ---------------- gambar prediksi di resolusi asli (10 m), gaya sama dengan plot klasifikasi RF ----------------
CMAP_RF = ListedColormap(["#2ecc71", "#e74c3c"])                          # 0 = hijau, 1 = merah
CMAP_CHG_10M = ListedColormap(["#ecf0f1", "#bdc3c7", "#e67e22", "#2980b9"])  # tetap 0, tetap 1, bertambah, berkurang
NATIVE_MAPS = {}


def native_annual_map(kec_id, year):
    """Peta tahunan resolusi asli (10 m), dibuat persis seperti cache (komposit + konsistensi waktu), hanya tanpa resize."""
    if kec_id not in NATIVE_MAPS:
        kota, kec = kec_id.split("__")
        if COMPOSITE_MODE == "mayoritas":
            years, maps, _ = build_annual_majority(kota, kec, COMPOSITE_MONTHS)
        else:
            years, maps = build_annual_composite(kota, kec, allowed_months=COMPOSITE_MONTHS)
        if TEMPORAL_CONSISTENCY:
            maps = apply_temporal_consistency(maps)
        NATIVE_MAPS[kec_id] = dict(zip(years, maps))
    return NATIVE_MAPS[kec_id][year]


def to_native(coarse, shape):
    """Sel 64x64 dikembalikan ke ukuran raster asli (nearest), kebalikan dari resize_to_patch."""
    return sk_resize(coarse, shape, order=0, preserve_range=True, anti_aliasing=False)


def native_forecast(native_last, coarse_last, coarse_pred):
    """Prediksi 10 m = peta aktual terakhir (10 m) + perubahan yang diprediksi model per sel 64x64.
    Sel yang menurut model tidak berubah memakai detail asli 2025; sel yang berubah diganti seluruhnya."""
    up_last = to_native(coarse_last, native_last.shape)
    up_pred = to_native(coarse_pred, native_last.shape)
    out = native_last.copy()
    valid = native_last != LABEL_NODATA
    out[valid & (up_last == 0) & (up_pred == 1)] = 1
    out[valid & (up_last == 1) & (up_pred == 0)] = 0
    return out


def km2(m, v):
    return float(np.sum(m == v)) * PIXEL_AREA_KM2


def show_rf(ax, m, title):
    ax.imshow(np.ma.masked_equal(m, LABEL_NODATA), cmap=CMAP_RF, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(handles=[Patch(facecolor="#2ecc71", edgecolor="black", label=f"Non-terbangun: {km2(m, 0):.2f} km²"),
                       Patch(facecolor="#e74c3c", edgecolor="black", label=f"Terbangun: {km2(m, 1):.2f} km²")],
              title="Kategori Tutupan Lahan", loc="lower right", fontsize=8.5, title_fontsize=9,
              frameon=True, framealpha=0.9, facecolor="white", edgecolor="gray")
    ax.axis("off")


for kec_id in PLOT_KECS:
    i = KEC_IDS.index(kec_id)
    bf, act = M_BEFORE[i], M_TEST[i]
    pred = (test_probs[0][i] > THRESHOLD).astype(np.float32)
    valid = (bf != -1) & (act != -1)
    err = np.zeros_like(act)
    err[valid & (bf == 0) & (act == 1) & (pred == 1)] = 1
    err[valid & (bf == 0) & (act == 1) & (pred == 0)] = 2
    err[valid & (bf == 0) & (act == 0) & (pred == 1)] = 3

    fig, ax = plt.subplots(1, 4, figsize=(16, 4.4))
    show_bin(ax[0], bf, f"Aktual {ORIGIN}")
    show_bin(ax[1], act, f"Aktual {ty}")
    show_bin(ax[2], np.where(act == -1, -1, pred), f"Prediksi {ty}")
    ax[3].imshow(np.ma.masked_where(~valid, err), cmap=CMAP_ERR, vmin=0, vmax=3, interpolation="nearest")
    ax[3].set_title("Galat perubahan", fontsize=10); ax[3].axis("off")
    legend_below(ax[3], [("#2e9e4f", "Tertangkap"), ("#f0a030", "Terlewat"), ("#d6372c", "Salah alarm")])
    fig.suptitle(f"Test {ty}: {kec_id}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_plot_test_{kec_id}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    m = evaluate_predictions(bf[None], act[None], test_probs[0][i][None])
    print(f"{kec_id}, test {ty}: akurasi {m['oa']:.4f} (persistensi {m['persist_oa']:.4f}) | "
          f"IoU {m['iou']:.4f} (persistensi {m['persist_iou']:.4f}) | F1 {m['f1']:.4f} | FoM {m['fom']:.4f}")
    lbl, col = ["aktual 0", "aktual 1"], ["prediksi 0", "prediksi 1"]
    print("Confusion matrix model:\n" + pd.DataFrame(confusion_from_maps(bf[None], act[None], pred[None]), lbl, col).to_string())
    print("Confusion matrix persistensi:\n" + pd.DataFrame(confusion_from_maps(bf[None], act[None], (bf == 1)[None]), lbl, col).to_string())

    # ---- perbandingan 2025 / 2026 / 2027 di resolusi 10 m ----
    native_last = native_annual_map(kec_id, LAST_OBSERVED_YEAR)
    native_preds = [native_forecast(native_last, M_LAST[i], fc_bins[s][i]) for s in range(N_FORECAST)]
    h, w = native_last.shape
    cell = f"{h / PATCH_SIZE * 10:.0f}×{w / PATCH_SIZE * 10:.0f} m"

    chg = np.zeros(native_last.shape, np.uint8)
    chg[native_last == 1] = 1
    chg[(native_last == 0) & (native_preds[0] == 1)] = 2
    chg[(native_last == 1) & (native_preds[0] == 0)] = 3

    fig, ax = plt.subplots(1, N_FORECAST + 2, figsize=(6 * (N_FORECAST + 2), 6), dpi=300)
    show_rf(ax[0], native_last, f"1. Aktual {LAST_OBSERVED_YEAR}\n(resolusi 10 m)")
    for s, fy in enumerate(FORECAST_YEARS):
        show_rf(ax[s + 1], native_preds[s], f"{s + 2}. Prediksi {fy}\n(ConvLSTM)")
    ax[-1].imshow(np.ma.masked_where(native_last == LABEL_NODATA, chg), cmap=CMAP_CHG_10M, vmin=0, vmax=3,
                  interpolation="nearest")
    ax[-1].set_title(f"{N_FORECAST + 2}. Perubahan {LAST_OBSERVED_YEAR} → {FORECAST_YEARS[0]}\n(per sel model {cell})",
                     fontsize=11, fontweight="bold")
    ax[-1].legend(handles=[Patch(facecolor="#ecf0f1", edgecolor="black", label="Tetap non-terbangun"),
                           Patch(facecolor="#bdc3c7", edgecolor="black", label="Tetap terbangun"),
                           Patch(facecolor="#e67e22", edgecolor="black", label=f"Bertambah: {km2(chg, 2):.2f} km²"),
                           Patch(facecolor="#2980b9", edgecolor="black", label=f"Berkurang: {km2(chg, 3):.2f} km²")],
                  title="Perubahan", loc="lower right", fontsize=8.5, title_fontsize=9,
                  frameon=True, framealpha=0.9, facecolor="white", edgecolor="gray")
    ax[-1].axis("off")
    kota, kec = kec_id.split("__")
    plt.suptitle(f"Perbandingan Lahan Terbangun {LAST_OBSERVED_YEAR}–{FORECAST_YEARS[-1]} — Kecamatan "
                 f"{kec.replace('_', ' ').title()}, {kota.replace('_', ' ').title()}", fontsize=13, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_plot_prediksi_{kec_id}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    r = pred_df.iloc[i]
    print(f"Luas terbangun 10 m (km2): {LAST_OBSERVED_YEAR} {km2(native_last, 1):.3f} | "
          + " | ".join(f"{fy} {km2(native_preds[s], 1):.3f}" for s, fy in enumerate(FORECAST_YEARS)))
    print(f"Luas terbangun versi 64x64 (tabel pred_df): {LAST_OBSERVED_YEAR} {r[f'luas_{LAST_OBSERVED_YEAR}_resize_km2']:.3f} | "
          + " | ".join(f"{fy} {r[f'prediksi_{fy}_km2']:.3f}" for fy in FORECAST_YEARS))
    print("-" * 80)


## 10. Perbandingan v7 vs v8

Sel ini membaca hasil v7 dari `V7_OUTPUT_DIR` (kalau ada) dan menaruhnya di samping hasil v8.

Label v7 dan v8 sama persis (cache yang sama), jadi angkanya bisa dibandingkan langsung. Perbedaannya hanya dua: (1) v8 memakai layer ConvLSTM yang ditulis sendiri, dengan nilai awal bobot acak yang berbeda dari v7, dan (2) di v8 jumlah layer ikut dipilih lewat grid search. Selisih kecil bisa saja hanya karena nilai awal acak yang berbeda.

In [ ]:
def load_csv(path):
    return pd.read_csv(path) if os.path.exists(path) else None


def change_by_region(pred):
    c25 = [c for c in pred.columns if c.startswith(f"luas_{LAST_OBSERVED_YEAR}") and c.endswith("resize_km2")][0]
    c26 = [c for c in pred.columns if str(FORECAST_YEARS[0]) in c][0]
    g = pred.groupby("kota")[[c25, c26]].sum()
    g.loc["TOTAL"] = g.sum()
    return ((g[c26] / g[c25] - 1) * 100).rename(f"perubahan {LAST_OBSERVED_YEAR}→{FORECAST_YEARS[0]} (%)")


KOLOM = ["akurasi", "precision", "recall", "f1", "iou", "fom", "MAE_km2", "bias_total_%"]
v7_test = load_csv(os.path.join(V7_OUTPUT_DIR, f"{MODEL_NAME}_evaluasi_test.csv"))
v7_pred = load_csv(os.path.join(V7_OUTPUT_DIR, f"{MODEL_NAME}_prediksi_luas_kecamatan.csv"))
v7_cfg_path = os.path.join(V7_OUTPUT_DIR, f"{MODEL_NAME}_best_config.json")

print("=== Konfigurasi terpilih ===")
cfgs = {"v8": {**BEST, "epochs": EPOCHS_REFIT}}
if os.path.exists(v7_cfg_path):
    with open(v7_cfg_path) as f:
        cfgs = {"v7": {k: v for k, v in json.load(f).items() if k in GRID_KEYS + ["epochs"]}, **cfgs}
display(pd.DataFrame(cfgs))

print("\n=== Test: model v7 vs v8 (label sama) ===")
if v7_test is not None:
    display(pd.concat({"v7": v7_test.set_index("tahun")[KOLOM], "v8": test_df.set_index("tahun")[KOLOM]}).round(4))
else:
    print(f"(hasil v7 tidak ditemukan di {V7_OUTPUT_DIR}, hanya v8 yang ditampilkan)")
    display(test_df.set_index("tahun")[KOLOM].round(4))

print(f"\n=== Prediksi {FORECAST_YEARS[0]}: perubahan luas terbangun per wilayah ===")
cmp = {"v8": change_by_region(pred_df)}
if v7_pred is not None:
    cmp = {"v7": change_by_region(v7_pred), **cmp}
display(pd.DataFrame(cmp).round(2))

col25, col26 = f"luas_{LAST_OBSERVED_YEAR}_resize_km2", f"prediksi_{FORECAST_YEARS[0]}_km2"
turun = pred_df[pred_df[col26] < 0.8 * pred_df[col25]]
print(f"\nKecamatan yang diprediksi turun lebih dari 20% di {FORECAST_YEARS[0]} (v8): {len(turun)}",
      list(turun["kecamatan"]) if len(turun) else "")


## Catatan untuk pembahasan

1. **Label berasal dari klasifikasi RF dengan acuan NDBI**, bukan pengamatan lapangan. ConvLSTM memprediksi peta label itu, termasuk kesalahannya.
2. **Label tahunan dibersihkan** dengan komposit mayoritas dan aturan konsistensi waktu. Keduanya bagian dari metode dan perlu ditulis, termasuk konsekuensinya: model dievaluasi terhadap label yang sudah dibersihkan, dan aturan konsistensi membuat perubahan terbangun → nonterbangun hampir tidak mungkin terjadi. Tahun yang tidak punya tanggal di rentang itu tetap memakai semua tanggal.
3. **Resize ke 64×64** menghilangkan detail (lihat tabel keutuhan resize). Alternatifnya, membagi peta resolusi asli menjadi blok-blok tanpa mengubah skala, seperti Boulila dkk. (2021) dengan blok 256×256.
4. **Data waktu sedikit** (10 tahun, validasi hanya 1 tahun), jadi selisih kecil antar kombinasi grid bisa kebetulan.
5. **Prediksi 2027 rekursif**, galat 2026 ikut terbawa. Hasil test 2025 (2 langkah) menggambarkan besarnya.

6. **Implementasi.** Rumus ConvLSTM (Persamaan 2.39–2.44) ditulis sendiri di `ConvLSTMManual`; konvolusi, gradien, Adam, BatchNormalization, dan Conv2D memakai TensorFlow/Keras. Sebutkan versinya di skripsi (`print(tf.__version__)`). Kesamaan dengan `ConvLSTM2D` Keras dibuktikan di bagian 3b.
7. **Jumlah layer dipilih lewat grid search** (1, 2, 3), karena literatur tidak memberi satu angka baku: Shi dkk. (2015) menguji 1–3 layer, Boulila dkk. (2021) memakai 4 layer, O'Neill dkk. (2025) memakai 1 layer, dan Durrani dkk. (2023) menjadikannya hyperparameter.

**Rujukan:** Boulila dkk. (2021), *Ecological Informatics* 64:101325 (arXiv:2103.01695); Shi dkk. (2015), NeurIPS (arXiv:1506.04214); Durrani dkk. (2023), *PLOS ONE* 18(2), doi:10.1371/journal.pone.0275653; O'Neill dkk. (2025), *Land* 14(8):1630, doi:10.3390/land14081630; Pontius dkk. (2008), *Annals of Regional Science* 42:11–37; Cerqueira, Torgo & Mozetič (2020), *Machine Learning* 109 (arXiv:1905.11744).